In [1]:
from pathlib import Path
from typing import List, Tuple, Dict, Any, Optional
import json, random, itertools, os, gc, statistics as st
import seqeval

import numpy as np
import pandas as pd
from datasets import Dataset, DatasetDict, concatenate_datasets
from sklearn.model_selection import KFold, train_test_split
from collections import Counter
from sentence_transformers import SentenceTransformer
from sklearn.neighbors import BallTree
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity, pairwise_distances
from sklearn.cluster import KMeans
from dataclasses import dataclass, field

import torch
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
)

from seqeval.metrics import f1_score, classification_report
from evaluate import load as load_metric
from tqdm.auto import tqdm

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def _get(d: Dict[str, Any], k: str, default=None):
    return d.get(k, default)


@dataclass
class NERMetricsCollector:
    overall_rows: List[Dict[str, Any]] = field(default_factory=list)
    label_rows: List[Dict[str, Any]] = field(default_factory=list)
    meta: Dict[str, Any] = field(
        default_factory=dict
    )  # ex.: nome do modelo, dataset, etc.

    def record(
        self,
        split_name: Any,
        metrics: Dict[str, Any],
        extras: Optional[Dict[str, Any]] = None,
    ):
        """
        Registra os resultados de um split.
        - split_name: pode ser string, int, tupla... será convertido para string
        - metrics: dict retornado pelo seu train/eval
        - extras: (opcional) dict com metadados (seed, versão, etc.)
        """
        split_str = str(split_name)

        # ------------------------------
        # Tabela 1: métricas gerais
        # ------------------------------
        overall = {
            "split": split_str,
            "eval_loss": _get(metrics, "eval_loss"),
            "overall_precision": _get(metrics, "eval_overall_precision"),
            "overall_recall": _get(metrics, "eval_overall_recall"),
            "overall_f1": _get(metrics, "eval_overall_f1"),
            "overall_accuracy": _get(metrics, "eval_overall_accuracy"),
            "f1_micro": _get(metrics, "eval_f1_micro"),
            "f1_macro": _get(metrics, "eval_f1_macro"),
            "f1_weighted": _get(metrics, "eval_f1_weighted"),
            "runtime_s": _get(metrics, "eval_runtime"),
            "samples_per_sec": _get(metrics, "eval_samples_per_second"),
            "steps_per_sec": _get(metrics, "eval_steps_per_second"),
            "epoch": _get(metrics, "epoch"),
        }

        self.overall_rows.append(overall)

        # ------------------------------
        # Tabela 2: métricas por rótulo
        # ------------------------------
        # Regra: qualquer entrada do dict que seja outro dict contendo
        # precision/recall/f1/number é tratada como rótulo.
        for k, v in metrics.items():
            if isinstance(v, dict) and {"precision", "recall", "f1", "number"} <= set(
                v.keys()
            ):
                self.label_rows.append(
                    {
                        "split": split_str,
                        "label": k.replace(
                            "eval_", ""
                        ),  # remove prefixo "eval_" para ficar limpo
                        "precision": v["precision"],
                        "recall": v["recall"],
                        "f1": v["f1"],
                        "support": v["number"],
                    }
                )

    # Comentário: retorna DataFrames prontos para inspeção ou export
    def to_dataframes(self):
        df_overall = pd.DataFrame(self.overall_rows)
        df_labels = pd.DataFrame(self.label_rows)
        return df_overall, df_labels

    # Comentário: exporta dois CSVs (UTF-8 com BOM para abrir liso no Excel)
    def to_csv(self, base_name: str = "ner"):
        df_overall, df_labels = self.to_dataframes()
        df_overall.to_csv(
            f"{base_name}_overall_metrics.csv", index=False, encoding="utf-8-sig"
        )
        df_labels.to_csv(
            f"{base_name}_label_metrics.csv", index=False, encoding="utf-8-sig"
        )
        return f"{base_name}_overall_metrics.csv", f"{base_name}_label_metrics.csv"


# --- Cria (ou reaproveita) um coletor global ---
if "ner_collector" not in globals():
    ner_collector = NERMetricsCollector()

# Configuração e Verificação Inicial

In [ ]:
SEED_GLOBAL = 42
random.seed(SEED_GLOBAL)
np.random.seed(SEED_GLOBAL)
torch.manual_seed(SEED_GLOBAL)
MODEL_NAME = "albert/albert-base-v2"


In [4]:
def read_conll_corenlp(path):
    sents, tags = [], []
    cur_tokens, cur_tags = [], []
    with open(path, encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                if cur_tokens:
                    sents.append(cur_tokens)
                    tags.append(cur_tags)
                    cur_tokens, cur_tags = [], []
                continue
            # Formato típico: token \t LABEL
            parts = line.split("\t")
            if len(parts) == 1:
                token, label = parts[0], "O"  # fallback, se vier sem label
            else:
                token, label = parts[0], parts[-1]
            cur_tokens.append(token)
            cur_tags.append(label)
    if cur_tokens:
        sents.append(cur_tokens)
        tags.append(cur_tags)
    return {"tokens": sents, "ner_tags_str": tags}


train = read_conll_corenlp("../data/bgs.3class.geo-training-data.txt")
test = read_conll_corenlp("../data/bgs.3class.geo-testing-data.txt")

In [5]:
labels = sorted(
    {lab for seq in train["ner_tags_str"] + test["ner_tags_str"] for lab in seq}
)
label2id = {l: i for i, l in enumerate(labels)}
id2label = {i: l for l, i in label2id.items()}

In [6]:
label2id

{'BIOZONE': 0, 'CHRONOSTRAT': 1, 'LEXICON': 2, 'O': 3}

In [7]:
id2label

{0: 'BIOZONE', 1: 'CHRONOSTRAT', 2: 'LEXICON', 3: 'O'}

In [8]:
def to_ids(batch):
    batch["ner_tags"] = [seq for seq in batch["ner_tags_str"]]
    return batch


ds_train = Dataset.from_dict(train).map(to_ids, batched=True)
ds_test = Dataset.from_dict(test).map(to_ids, batched=True)

Map: 100%|██████████| 594/594 [00:00<00:00, 41098.24 examples/s]


In [9]:
_split = ds_train.train_test_split(test_size=0.15, seed=42)
ds_train_split = _split["train"]
ds_dev_split = _split["test"]

geo_ds = DatasetDict(train=ds_train_split, dev=ds_dev_split, test=ds_test)

cols = geo_ds["train"].column_names
geo_ds["dev"] = geo_ds["dev"].select_columns(cols)
geo_ds["test"] = geo_ds["test"].select_columns(cols)

geo_ds["train"] = geo_ds["train"].add_column("split", ["train"] * len(geo_ds["train"]))
geo_ds["dev"] = geo_ds["dev"].add_column("split", ["dev"] * len(geo_ds["dev"]))
geo_ds["test"] = geo_ds["test"].add_column("split", ["test"] * len(geo_ds["test"]))

geo_full = concatenate_datasets([geo_ds["train"], geo_ds["dev"], geo_ds["test"]])   

Flattening the indices: 100%|██████████| 380/380 [00:00<00:00, 101035.53 examples/s]


In [10]:
geo_ds

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags_str', 'ner_tags', 'split'],
        num_rows: 2151
    })
    dev: Dataset({
        features: ['tokens', 'ner_tags_str', 'ner_tags', 'split'],
        num_rows: 380
    })
    test: Dataset({
        features: ['tokens', 'ner_tags_str', 'ner_tags', 'split'],
        num_rows: 594
    })
})

In [11]:
# lista de rótulos (ordem alfabética garante consistência entre runs)
label_list = sorted({l for sent in geo_full["ner_tags"] for l in sent})
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
NUM_LABELS = len(label_list)

In [12]:
id2label

{0: 'BIOZONE', 1: 'CHRONOSTRAT', 2: 'LEXICON', 3: 'O'}

In [13]:
NUM_LABELS

4

# Splits

In [14]:
def loc_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    ngram: int = 4,
    seed: int = 42,
) -> DatasetDict:
    """
    Divide por sobreposição léxica (n-gram Jaccard).
    Frações independentes para teste e validação.
    """
    docs = [" ".join(toks) for toks in dataset["tokens"]]

    vect = CountVectorizer(
        analyzer="word", ngram_range=(ngram, ngram), binary=True
    ).fit(docs)
    X = vect.transform(docs)
    bin_counts = X.sum(axis=1).A1

    sim = cosine_similarity(X, dense_output=False)
    k = 5
    topk = np.zeros(len(dataset))
    for i in range(sim.shape[0]):
        row = sim.getrow(i).toarray()[0]
        idx = np.argpartition(-row, range(1, k + 1))[1 : k + 1]
        inter = row[idx] * bin_counts[i]
        uni = bin_counts[i] + bin_counts[idx] - inter
        topk[i] = (inter / uni).mean()

    order = np.argsort(topk)  # baixo → alto overlap
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[:n_test]
    val_idx = order[n_test : n_test + n_val]
    train_idx = order[n_test + n_val :]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [15]:
def semantic_cluster_split(
    dataset: Dataset,
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    k: int | None = None,
    seed: int = 42,
) -> DatasetDict:
    """
    Clusters SBERT → reserva clusters distantes para test/val.
    """
    if k is None:
        k = int(np.sqrt(len(dataset)))

    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    embeddings = sbert.encode(
        [" ".join(t) for t in tqdm(dataset["tokens"])],
        batch_size=64,
        show_progress_bar=False,
    )

    km = KMeans(n_clusters=k, random_state=seed, n_init=10).fit(embeddings)
    labels = km.labels_
    centroids = km.cluster_centers_

    global_center = embeddings.mean(0, keepdims=True)
    dists = pairwise_distances(centroids, global_center).flatten()

    clusters_sorted = np.argsort(-dists)  # mais distantes primeiro
    test_clusters, val_clusters = set(), set()
    total_test = total_val = 0
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    for c in clusters_sorted:
        size = np.sum(labels == c)
        if total_test < n_test:  # preenche primeiro o teste
            test_clusters.add(c)
            total_test += size
        elif total_val < n_val:  # depois a validação
            val_clusters.add(c)
            total_val += size
        if total_test >= n_test and total_val >= n_val:
            break

    test_idx = np.where([lbl in test_clusters for lbl in labels])[0]
    val_idx = np.where([lbl in val_clusters for lbl in labels])[0]
    train_idx = np.where(
        [lbl not in test_clusters and lbl not in val_clusters for lbl in labels]
    )[0]

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [16]:
def difficulty_scores(dataset: Dataset) -> np.ndarray:
    length = np.array([len(tok) for tok in dataset["tokens"]], dtype=float)
    length = (length - length.mean()) / length.std()

    dens = []
    for labels in dataset["ner_tags"]:
        non_o = sum(1 for l in labels if l != "O")
        dens.append(non_o / len(labels))
    dens = np.array(dens)
    dens = (dens - dens.mean()) / dens.std()

    ent_types, freq = [], Counter()
    for labels in dataset["ner_tags"]:
        types = [l[2:] for l in labels if l != "O"]
        ent_types.append(types[0] if types else "NONE")
    freq.update(ent_types)
    rarity = np.array([1 / freq[t] for t in ent_types])
    rarity = (rarity - rarity.mean()) / rarity.std()

    return length + dens + rarity


def reverse_curriculum_split(
    dataset: Dataset, pct_test: float = 0.20, pct_val: float = 0.10, seed: int = 42
) -> DatasetDict:
    scores = difficulty_scores(dataset)
    order = np.argsort(scores)  # easy→hard
    n_test = int(len(dataset) * pct_test)
    n_val = int(len(dataset) * pct_val)

    test_idx = order[-n_test:]  # hardest
    val_idx = order[-(n_test + n_val) : -n_test]
    train_idx = order[: -(n_test + n_val)]

    rng = np.random.RandomState(seed)
    rng.shuffle(train_idx)
    rng.shuffle(val_idx)
    rng.shuffle(test_idx)

    return DatasetDict(
        {
            "train": dataset.select(train_idx),
            "val": dataset.select(val_idx),
            "test": dataset.select(test_idx),
        }
    )

In [17]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42) -> DatasetDict:
    """Testa só sentenças ≥ máx(len_train)."""
    sent_lens = np.array([len(t) for t in dataset["tokens"]])
    # separação inicial train/dev (randômica estratificando por tamanho grosso)
    idx_all   = np.arange(len(dataset))
    train_idx, temp_idx = train_test_split(idx_all,
                                           test_size=pct_test + pct_val,
                                           stratify=(sent_lens//5),  # bin len
                                           random_state=seed)
    # define longo-threshold como tamanho máx. do treino
    max_train_len = sent_lens[train_idx].max()
    # test = sentenças > threshold.  Caso falte/ sobre exemplos, ajusta.
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]
    # completa ou reduz para atingir pct_test
    need = int(pct_test*len(dataset)) - len(test_idx)
    if need > 0:
        test_idx.extend(resto_idx[:need])
        val_idx = resto_idx[need:]
    else:
        val_keep = int(pct_val*len(dataset))
        val_idx  = resto_idx[:val_keep]
        test_idx = test_idx[: int(pct_test*len(dataset))]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 2) Heuristic Rare-Words ----------------------------------
def heur_rare_split(dataset: Dataset,
                    pct_test: float = 0.20,
                    pct_val : float = 0.10,
                    seed: int = 42) -> DatasetDict:
    """Testa frases que contenham palavras do quintil + raro."""
    # contagem de frequência de token
    freqs = Counter(w for toks in dataset["tokens"] for w in toks)
    # define rareza: 20 % mais raras
    thresh = np.quantile(list(freqs.values()), 0.20)
    rare_set = {w for w,c in freqs.items() if c <= thresh}
    is_rare = np.array([any(w in rare_set for w in toks)
                        for toks in dataset["tokens"]])
    rare_idx   = np.where(is_rare)[0]
    common_idx = np.where(~is_rare)[0]
    # garante proporções desejadas
    n_test = int(pct_test*len(dataset))
    n_val  = int(pct_val *len(dataset))
    rng = np.random.default_rng(seed)
    test_idx = rng.choice(rare_idx, size=min(len(rare_idx), n_test),
                          replace=False)
    resto_idx = [i for i in rare_idx if i not in test_idx] + list(common_idx)
    val_idx  = rng.choice(resto_idx, size=n_val, replace=False)
    train_idx = [i for i in resto_idx if i not in val_idx]
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 3) Standard (80-10-10) -----------------------------------
def std_split(dataset: Dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42) -> DatasetDict:
    """Split aleatório estratificado por comprimento (PTB-like)."""
    idx = np.arange(len(dataset))
    strat = (np.array([len(t) for t in dataset["tokens"]]) // 5)
    train_idx, temp_idx = train_test_split(idx, test_size=pct_test+pct_val,
                                          stratify=strat, random_state=seed)
    val_rel = pct_val / (pct_test+pct_val)
    val_idx, test_idx = train_test_split(temp_idx, test_size=1-val_rel,
                                         stratify=strat[temp_idx],
                                         random_state=seed)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

# ---------- 4) Adversarial (approx. Wasserstein) ---------------------
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Seleciona k frases 'mais distantes' recursivamente (BallTree + W₂)
    para compor o teste, lembrando Alg.-1 de Søgaard et al.【turn6file4】.
    """
    # SBERT embed (rápido na GPU / aceitável CPU)
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(toks) for toks in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    # BallTree para vizinhança eficiente
    tree = BallTree(emb, leaf_size=40)
    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)
    print(f"Selecionando {int(pct_test*len(dataset))} sentenças para teste...")
    while len(test_idx) < int(pct_test*len(dataset)):
        print(f"  {len(test_idx)} selecionadas...")
        # amostra candidata: ponto + longe do centro
        center = emb[list(idx_pool)].mean(0, keepdims=True)
        dists, _ = tree.query(center, k=len(idx_pool))
        farthest = list(idx_pool)[int(dists.argmax())]
        # pega-se farest e seus k-NN mais próximos  ⇒ aumenta diversidade
        nn = tree.query([emb[farthest]], k=k, return_distance=False)[0]
        for j in nn:
            if j in idx_pool and len(test_idx) < int(pct_test*len(dataset)):
                test_idx.append(j)
                idx_pool.remove(j)
    # retira val
    val_size = int(pct_val*len(dataset))
    val_idx  = rng.choice(list(idx_pool), size=val_size, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)
    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [18]:
def heur_len_split(dataset: Dataset,
                   pct_test: float = 0.20,
                   pct_val : float = 0.10,
                   seed: int = 42,
                   bin_size: int = 10) -> DatasetDict:
    """
    Teste = sentenças mais longas que max(len(train)).
    Robusto a datasets pequenos: se estratificação falhar, usa split aleatório.
    """
    rng = np.random.default_rng(seed)
    idx_all   = np.arange(len(dataset))
    sent_lens = np.array([len(t) for t in dataset["tokens"]])

    # ------ 1) tenta split estratificado por baldes -------------------------
    strat = (sent_lens // bin_size)
    try:
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            stratify=strat,
            random_state=seed,
        )
    except ValueError:                       # classes com 1 amostra
        train_idx, temp_idx = train_test_split(
            idx_all,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # ------ 2) escolhe test = len > max(train) ------------------------------
    max_train_len = sent_lens[train_idx].max()
    test_idx  = [i for i in temp_idx if sent_lens[i] > max_train_len]
    resto_idx = [i for i in temp_idx if i not in test_idx]

    # garante tamanhos exatos
    n_test_desired = int(pct_test * len(dataset))
    n_val_desired  = int(pct_val  * len(dataset))

    # completa teste se ficou pequeno
    if len(test_idx) < n_test_desired:
        extra = rng.choice(resto_idx,
                           size=n_test_desired - len(test_idx),
                           replace=False)
        test_idx.extend(extra)
        resto_idx = [i for i in resto_idx if i not in extra]

    # define validação
    val_idx  = rng.choice(resto_idx, size=n_val_desired, replace=False)
    train_idx = [i for i in idx_all if i not in test_idx and i not in val_idx]

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [19]:
def std_split(dataset,
              pct_test: float = 0.10,
              pct_val : float = 0.10,
              seed: int = 42,
              bin_size: int = 5) -> DatasetDict:
    """
    Split 80-10-10 robusto.
      • Tenta estratificar por comprimento // bin_size.
      • Se houver classes com <2 amostras, recua p/ split aleatório.
    """
    idx  = np.arange(len(dataset))
    bins = (np.array([len(t) for t in dataset["tokens"]]) // bin_size)

    try:
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            stratify=bins,
            random_state=seed,
        )
    except ValueError:                       # classes muito pequenas
        train_idx, temp_idx = train_test_split(
            idx,
            test_size=pct_test + pct_val,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    # fraciona temp em val / test mantendo proporção desejada
    val_share = pct_val / (pct_test + pct_val)
    try:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            stratify=bins[temp_idx],
            random_state=seed,
        )
    except ValueError:
        val_idx, test_idx = train_test_split(
            temp_idx,
            test_size=1 - val_share,
            shuffle=True,
            random_state=seed,
            stratify=None,
        )

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [20]:
def adversarial_split(dataset: Dataset,
                      pct_test: float = 0.20,
                      pct_val : float = 0.10,
                      k: int = 5,
                      seed: int = 42) -> DatasetDict:
    """
    Versão robusta: nunca “trava” antes de atingir n_test.
    Seleciona blocos de k sentenças mais distantes do centro iterativamente.
    """
    # -------------------------------- embeds -------------------------------
    sbert = SentenceTransformer(
        "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    )
    emb = sbert.encode([" ".join(t) for t in dataset["tokens"]],
                       batch_size=64, show_progress_bar=False)
    tree = BallTree(emb, leaf_size=40)

    n_test = int(pct_test * len(dataset))
    n_val  = int(pct_val  * len(dataset))

    idx_pool = set(range(len(dataset)))
    test_idx = []
    rng = np.random.default_rng(seed)

    print(f"Selecionando {n_test} sentenças para teste…")
    step = 0
    while len(test_idx) < n_test and idx_pool:
        if step % 5 == 0:
            print(f"  {len(test_idx)} selecionadas…")
        step += 1

        pool_list = list(idx_pool)
        center = emb[pool_list].mean(0, keepdims=True)

        # distância euclidiana ao centro global restante
        dists, _ = tree.query(center, k=len(pool_list))
        farthest_local_idx = int(dists.argmax())
        farthest_global_idx = pool_list[farthest_local_idx]

        # número efetivo de vizinhos
        k_eff = min(k, len(idx_pool))
        nn = set(tree.query([emb[farthest_global_idx]],
                            k=k_eff,
                            return_distance=False)[0])

        # adiciona vizinhos ainda não selecionados
        for j in nn:
            if j in idx_pool and len(test_idx) < n_test:
                test_idx.append(j)
                idx_pool.remove(j)

        # Se nada foi adicionado (pode acontecer quando sobram <k únicos)
        if farthest_global_idx not in test_idx:
            test_idx.append(farthest_global_idx)
            idx_pool.remove(farthest_global_idx)

    # ------------------------ validação e treino ---------------------------
    val_idx = rng.choice(list(idx_pool), size=n_val, replace=False)
    idx_pool -= set(val_idx)
    train_idx = list(idx_pool)

    return DatasetDict({
        "train": dataset.select(train_idx),
        "val"  : dataset.select(val_idx),
        "test" : dataset.select(test_idx),
    })

In [21]:
def standard_split_conll(dataset, 
                         pct_test: float = 0.20,
                    pct_val : float = 0.10,
                    seed: int = 42):

    ds = DatasetDict({("val" if k == "dev" else k): v for k, v in geo_ds.items()})
    return ds

In [22]:
standard_split = standard_split_conll(geo_full)
# print('std')
# # random_splt = random_splits(geo_full)
# # print('random')
# heur_len = heur_len_split(geo_full)
# print("heur_len")
# heur_rare = heur_rare_split(geo_full)
# print("heur_rare")
# advers = adversarial_split(geo_full)
# print("advs")
# loc = loc_split(geo_full)
# print("loc")
# semantic = semantic_cluster_split(geo_full)
# print("semantic")
# reverse = reverse_curriculum_split(geo_full)
# print("reverse")

In [23]:
standard_split

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags_str', 'ner_tags', 'split'],
        num_rows: 2151
    })
    val: Dataset({
        features: ['tokens', 'ner_tags_str', 'ner_tags', 'split'],
        num_rows: 380
    })
    test: Dataset({
        features: ['tokens', 'ner_tags_str', 'ner_tags', 'split'],
        num_rows: 594
    })
})

# Experimentos

In [24]:
from sklearn.metrics import f1_score as skl_f1

In [25]:
def train_ner_with_split(
    dataset: Dataset,
    split: str,  # "loc" | "semantic" | "reverse" | func
    pct_test: float = 0.20,
    pct_val: float = 0.10,
    model_ckpt: str = "Davlan/distilbert-base-multilingual-cased-ner-hrl",
    training_args_kwargs: dict | None = None,
    split_kwargs: dict | None = None,
    seed: int = 42,
):
    """
    Treina um modelo de NER usando a estratégia de split desejada.
    Retorna (trainer, métricas_test).
    """
    # 1. Escolhe função de split ------------------------------------------------
    if callable(split):
        split_fn = split
    else:
        _map = {
            "loc": loc_split,
            "reverse": reverse_curriculum_split,
            "semantic": semantic_cluster_split,
            "heur_len": heur_len_split,
            "heur_rare": heur_rare_split,
            "std": standard_split_conll,
            "advs": adversarial_split,
        }
        if split not in _map:
            raise ValueError(f"split='{split}' não reconhecido.")
        split_fn = _map[split]

    split_kwargs = split_kwargs or {}
    ds = split_fn(
        dataset, pct_test=pct_test, pct_val=pct_val, seed=seed, **split_kwargs
    )  # train/val/test

    # 2. Tokenizer e modelo -----------------------------------------------------
    label_list = sorted({l for labels in dataset["ner_tags"] for l in labels})
    label2id = {l: i for i, l in enumerate(label_list)}
    id2label = {i: l for l, i in label2id.items()}

    num_labels = len(label_list)
    tok = AutoTokenizer.from_pretrained(model_ckpt)
    model = AutoModelForTokenClassification.from_pretrained(
        model_ckpt,
        num_labels=num_labels,  # ← adapta o tamanho
        ignore_mismatched_sizes=True,  # ← descarta pesos velhos da head
    )
    model.config.id2label = id2label
    model.config.label2id = label2id

    # 3. Mapeamento label↔id ----------------------------------------------------
    # label_list = sorted(
    #     {l for labels in dataset["ner_tags"] for l in labels if l != "O"}
    # )

    # 4. Tokenização + alinhamento ---------------------------------------------
    # def tok_function(ex):
    #     return tok(
    #         ex["tokens"], is_split_into_words=True, truncation=True, padding=False
    #     )

    # def align_labels(ex):
    #     word_ids = ex.word_ids()
    #     labels = []
    #     for w in word_ids:
    #         if w is None:
    #             labels.append(-100)
    #         else:
    #             labels.append(label2id.get(ex["ner_tags"][w], 0))
    #     ex["labels"] = labels
    #     return ex

    # ds_tok = ds.map(tok_function, batched=True)
    # ds_tok = ds_tok.map(align_labels)

    def tokenize_and_align_labels(examples, label_all_tokens=False):
        tokenized = tok(examples["tokens"], is_split_into_words=True, truncation=True)

        labels_batch = []
        for i, word_labels in enumerate(examples["ner_tags"]):
            word_ids = tokenized.word_ids(batch_index=i)  # <- aqui sim
            label_ids = []
            previous_word_idx = None
            for word_idx in word_ids:
                if word_idx is None:
                    label_ids.append(-100)  # máscara
                elif word_idx != previous_word_idx:
                    label_ids.append(label2id[word_labels[word_idx]])
                else:
                    # marca sub-tokens; mude para `label2id[...]`
                    # se quiser repetir label em todos os sub-tokens
                    label_ids.append(
                        label2id[word_labels[word_idx]] if label_all_tokens else -100
                    )
                previous_word_idx = word_idx
            labels_batch.append(label_ids)

        tokenized["labels"] = labels_batch
        return tokenized

    ds_tok = ds.map(
        tokenize_and_align_labels, batched=True, remove_columns=ds["train"].column_names
    )
    # 5. Métrica (seqeval) ------------------------------------------------------
    seqeval = load_metric("seqeval")

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=-1)

        sent_preds, sent_labels = [], []  # p/ seqeval
        flat_preds, flat_labels = [], []  # p/ sklearn

        for p_row, l_row in zip(preds, labels):
            p_sent, l_sent = [], []
            for pi, li in zip(p_row, l_row):
                if li != -100:
                    lbl_true = id2label[li]
                    lbl_pred = id2label[pi]
                    p_sent.append(lbl_pred)
                    l_sent.append(lbl_true)
                    flat_preds.append(lbl_pred)
                    flat_labels.append(lbl_true)
            sent_preds.append(p_sent)
            sent_labels.append(l_sent)

        # métricas seqeval (micro F1 = overall_f1)
        seqeval_metrics = seqeval.compute(
            predictions=sent_preds,
            references=sent_labels,
        )

        # métricas sklearn
        f1_micro = skl_f1(flat_labels, flat_preds, average="micro", zero_division=0)
        f1_macro = skl_f1(flat_labels, flat_preds, average="macro", zero_division=0)
        f1_weighted = skl_f1(
            flat_labels, flat_preds, average="weighted", zero_division=0
        )

        return {
            **seqeval_metrics,  # overall_precision / recall / f1
            "f1_micro": f1_micro,
            "f1_macro": f1_macro,
            "f1_weighted": f1_weighted,
        }

    # 6. Args de treinamento ----------------------------------------------------
    args_defaults = dict(
        # output_dir=f"ner-{split}",
        # estratégia de avaliação + salvamento
        eval_strategy="epoch",  # novo nome (4.52+)
        save_strategy="no",
        # load_best_model_at_end=True,
        metric_for_best_model="overall_f1",
        per_device_train_batch_size=16,
        per_device_eval_batch_size=16,
        num_train_epochs=5,
        seed=seed,
        report_to="none",
    )

    if training_args_kwargs:
        args_defaults.update(training_args_kwargs)
    args = TrainingArguments(**args_defaults)

    data_collator = DataCollatorForTokenClassification(tokenizer=tok, padding=True)

    # 7. Trainer ---------------------------------------------------------------
    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=ds_tok["train"],
        eval_dataset=ds_tok["val"],
        tokenizer=tok,
        data_collator=data_collator,
        compute_metrics=compute_metrics,
    )

    trainer.train()

    # 8. Avaliação final --------------------------------------------------------
    test_metrics = trainer.evaluate(eval_dataset=ds_tok["test"])
    return trainer, test_metrics

In [26]:
splits = ["loc", "reverse", "semantic", "heur_len", "heur_rare", "std", "advs"]

In [27]:
results = {}
trainer_all = {}
s = splits[0]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(geo_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: loc


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([4]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([4, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 625/625 [00:00<00:00, 14113.64 examples/s]
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/torch/cuda/__init__.py:129: UserWarning: CUDA initialization: CUDA unknown error - this may be due to an incorrectly set up environment, e.g. changing env variable CUDA_VISIBLE_DEVICES after program start. Setting the available devices to be zero. (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:109

Epoch,Training Loss,Validation Loss,Exicon,Hronostrat,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,No log,0.015489,"{'precision': 0.9086294416243654, 'recall': 0.927461139896373, 'f1': 0.9179487179487179, 'number': 193}","{'precision': 0.9358288770053476, 'recall': 0.9831460674157303, 'f1': 0.958904109589041, 'number': 178}",0.921875,0.954178,0.937748,0.994313,0.994313,0.973223,0.994352
2,No log,0.012898,"{'precision': 0.9158415841584159, 'recall': 0.9585492227979274, 'f1': 0.9367088607594937, 'number': 193}","{'precision': 0.9831460674157303, 'recall': 0.9831460674157303, 'f1': 0.9831460674157303, 'number': 178}",0.947368,0.970350,0.958722,0.996538,0.996538,0.986267,0.996560
3,No log,0.016349,"{'precision': 0.8990384615384616, 'recall': 0.9689119170984456, 'f1': 0.9326683291770574, 'number': 193}","{'precision': 0.9725274725274725, 'recall': 0.9943820224719101, 'f1': 0.9833333333333333, 'number': 178}",0.933333,0.981132,0.956636,0.996291,0.996291,0.985479,0.996323
4,0.019600,0.014018,"{'precision': 0.9230769230769231, 'recall': 0.9326424870466321, 'f1': 0.9278350515463918, 'number': 193}","{'precision': 0.9565217391304348, 'recall': 0.9887640449438202, 'f1': 0.9723756906077348, 'number': 178}",0.939314,0.959569,0.949333,0.995549,0.995549,0.980021,0.995556
5,0.019600,0.015560,"{'precision': 0.905940594059406, 'recall': 0.9481865284974094, 'f1': 0.9265822784810126, 'number': 193}","{'precision': 0.9668508287292817, 'recall': 0.9831460674157303, 'f1': 0.9749303621169917, 'number': 178}",0.934726,0.964960,0.949602,0.995920,0.995920,0.982908,0.995937


/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LEXICON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: CHRONOSTRAT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LEXICON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: CHRONOSTRAT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LEXI

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LEXICON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: CHRONOSTRAT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: BIOZONE seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


F1 Macro: 0.89193771921218
F1 Micro: 0.9930506754118841
F1 Weighted: 0.9930581376489904
{'eval_loss': 0.02658062055706978, 'eval_EXICON': {'precision': 0.8964285714285715, 'recall': 0.9227941176470589, 'f1': 0.9094202898550725, 'number': 272}, 'eval_HRONOSTRAT': {'precision': 0.9538461538461539, 'recall': 0.9480122324159022, 'f1': 0.9509202453987731, 'number': 327}, 'eval_IOZONE': {'precision': 1.0, 'recall': 0.5, 'f1': 0.6666666666666666, 'number': 4}, 'eval_overall_precision': 0.9275123558484349, 'eval_overall_recall': 0.9336650082918739, 'eval_overall_f1': 0.9305785123966942, 'eval_overall_accuracy': 0.9930506754118841, 'eval_f1_micro': 0.9930506754118841, 'eval_f1_macro': 0.89193771921218, 'eval_f1_weighted': 0.9930581376489904, 'eval_runtime': 9.8174, 'eval_samples_per_second': 63.663, 'eval_steps_per_second': 4.074, 'epoch': 5.0}




0

In [28]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu": "0"})

In [29]:
!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [30]:
import time

In [31]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch

time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [32]:
results = {}
trainer_all = {}
s = splits[1]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(geo_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: reverse


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([4]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([4, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 625/625 [00:00<00:00, 8668.41 examples/s]
/tmp/ipykernel_382112/1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Exicon,Hronostrat,Iozone,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,No log,0.031579,"{'precision': 0.959349593495935, 'recall': 0.9315789473684211, 'f1': 0.9452603471295059, 'number': 380}","{'precision': 0.9570552147239264, 'recall': 0.9176470588235294, 'f1': 0.9369369369369369, 'number': 170}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 2}",0.958647,0.923913,0.940959,0.991898,0.991898,0.724843,0.991739
2,No log,0.039222,"{'precision': 0.9603399433427762, 'recall': 0.8921052631578947, 'f1': 0.9249658935879946, 'number': 380}","{'precision': 0.9578313253012049, 'recall': 0.9352941176470588, 'f1': 0.9464285714285714, 'number': 170}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 2}",0.959538,0.902174,0.929972,0.990258,0.990258,0.724145,0.990020
3,No log,0.039935,"{'precision': 0.9578651685393258, 'recall': 0.8973684210526316, 'f1': 0.9266304347826088, 'number': 380}","{'precision': 0.9705882352941176, 'recall': 0.9705882352941176, 'f1': 0.9705882352941176, 'number': 170}","{'precision': 1.0, 'recall': 0.5, 'f1': 0.6666666666666666, 'number': 2}",0.962049,0.918478,0.939759,0.991223,0.991223,0.895242,0.991103
4,0.020300,0.034491,"{'precision': 0.9516129032258065, 'recall': 0.9315789473684211, 'f1': 0.9414893617021277, 'number': 380}","{'precision': 0.9761904761904762, 'recall': 0.9647058823529412, 'f1': 0.9704142011834319, 'number': 170}","{'precision': 1.0, 'recall': 0.5, 'f1': 0.6666666666666666, 'number': 2}",0.959335,0.940217,0.949680,0.992670,0.992670,0.898284,0.992615
5,0.020300,0.036750,"{'precision': 0.9514824797843666, 'recall': 0.9289473684210526, 'f1': 0.9400798934753662, 'number': 380}","{'precision': 0.9704142011834319, 'recall': 0.9647058823529412, 'f1': 0.967551622418879, 'number': 170}","{'precision': 1.0, 'recall': 0.5, 'f1': 0.6666666666666666, 'number': 2}",0.957486,0.938406,0.947850,0.992670,0.992670,0.897633,0.992610


/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LEXICON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: CHRONOSTRAT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: BIOZONE seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/me

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LEXICON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: CHRONOSTRAT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: BIOZONE seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


F1 Macro: 0.8582112221882054
F1 Micro: 0.9918695831613702
F1 Weighted: 0.9917620490817763
{'eval_loss': 0.03474094718694687, 'eval_EXICON': {'precision': 0.9639293937068304, 'recall': 0.9429429429429429, 'f1': 0.9533206831119544, 'number': 1332}, 'eval_HRONOSTRAT': {'precision': 0.9693877551020408, 'recall': 0.9531772575250836, 'f1': 0.9612141652613828, 'number': 299}, 'eval_IOZONE': {'precision': 1.0, 'recall': 0.3333333333333333, 'f1': 0.5, 'number': 9}, 'eval_overall_precision': 0.965, 'eval_overall_recall': 0.9414634146341463, 'eval_overall_f1': 0.9530864197530864, 'eval_overall_accuracy': 0.9918695831613702, 'eval_f1_micro': 0.9918695831613702, 'eval_f1_macro': 0.8582112221882054, 'eval_f1_weighted': 0.9917620490817763, 'eval_runtime': 15.7337, 'eval_samples_per_second': 39.724, 'eval_steps_per_second': 2.542, 'epoch': 5.0}




0

In [33]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu": "0"})

In [34]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch


time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [35]:
results = {}
trainer_all = {}
s = splits[2]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(geo_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: semantic


100%|██████████| 3125/3125 [00:00<00:00, 1894925.55it/s]
Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([4]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([4, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 649/649 [00:00<00:00, 11678.94 examples/s]
/tmp/ipykernel_382112/1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Exicon,Hronostrat,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,No log,0.016028,"{'precision': 0.9079754601226994, 'recall': 1.0, 'f1': 0.9517684887459806, 'number': 148}","{'precision': 0.9437229437229437, 'recall': 0.9775784753363229, 'f1': 0.9603524229074891, 'number': 223}",0.928934,0.986523,0.956863,0.995216,0.995216,0.976905,0.995277
2,No log,0.013392,"{'precision': 0.9483870967741935, 'recall': 0.9932432432432432, 'f1': 0.9702970297029702, 'number': 148}","{'precision': 0.9523809523809523, 'recall': 0.9865470852017937, 'f1': 0.9691629955947136, 'number': 223}",0.950777,0.989218,0.969617,0.996773,0.996773,0.983773,0.996798
3,No log,0.015708,"{'precision': 0.9363057324840764, 'recall': 0.9932432432432432, 'f1': 0.9639344262295082, 'number': 148}","{'precision': 0.9519650655021834, 'recall': 0.9775784753363229, 'f1': 0.9646017699115046, 'number': 223}",0.945596,0.983827,0.964333,0.996440,0.996440,0.982029,0.996473
4,0.023200,0.015969,"{'precision': 0.930379746835443, 'recall': 0.9932432432432432, 'f1': 0.9607843137254902, 'number': 148}","{'precision': 0.9601769911504425, 'recall': 0.9730941704035875, 'f1': 0.9665924276169265, 'number': 223}",0.947917,0.981132,0.964238,0.996440,0.996440,0.981556,0.996471
5,0.023200,0.016106,"{'precision': 0.930379746835443, 'recall': 0.9932432432432432, 'f1': 0.9607843137254902, 'number': 148}","{'precision': 0.9601769911504425, 'recall': 0.9730941704035875, 'f1': 0.9665924276169265, 'number': 223}",0.947917,0.981132,0.964238,0.996440,0.996440,0.981556,0.996471


/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: CHRONOSTRAT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LEXICON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: CHRONOSTRAT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LEXICON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: CHRO

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: CHRONOSTRAT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LEXICON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


F1 Macro: 0.9834859653392103
F1 Micro: 0.9961443460449425
F1 Weighted: 0.9961612829231281
{'eval_loss': 0.015234186314046383, 'eval_EXICON': {'precision': 0.9438877755511023, 'recall': 0.9771784232365145, 'f1': 0.9602446483180428, 'number': 482}, 'eval_HRONOSTRAT': {'precision': 0.954983922829582, 'recall': 0.9705882352941176, 'f1': 0.9627228525121555, 'number': 306}, 'eval_overall_precision': 0.9481481481481482, 'eval_overall_recall': 0.9746192893401016, 'eval_overall_f1': 0.9612015018773468, 'eval_overall_accuracy': 0.9961443460449425, 'eval_f1_micro': 0.9961443460449425, 'eval_f1_macro': 0.9834859653392103, 'eval_f1_weighted': 0.9961612829231281, 'eval_runtime': 12.8165, 'eval_samples_per_second': 50.638, 'eval_steps_per_second': 3.199, 'epoch': 5.0}




0

In [36]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu": "0"})

In [37]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch
time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [38]:
results = {}
trainer_all = {}
s = splits[3]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(geo_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: heur_len


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([4]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([4, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 625/625 [00:00<00:00, 10689.98 examples/s]
/tmp/ipykernel_382112/1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Exicon,Hronostrat,Iozone,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,No log,0.020014,"{'precision': 0.9702602230483272, 'recall': 0.925531914893617, 'f1': 0.9473684210526315, 'number': 282}","{'precision': 0.9702380952380952, 'recall': 0.9878787878787879, 'f1': 0.978978978978979, 'number': 165}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 2}",0.970252,0.944321,0.957111,0.994600,0.994600,0.737252,0.994439
2,No log,0.011280,"{'precision': 0.9517241379310345, 'recall': 0.9787234042553191, 'f1': 0.965034965034965, 'number': 282}","{'precision': 0.9761904761904762, 'recall': 0.9939393939393939, 'f1': 0.984984984984985, 'number': 165}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 2}",0.960699,0.979955,0.970232,0.996625,0.996625,0.741907,0.996521
3,No log,0.013292,"{'precision': 0.9614035087719298, 'recall': 0.9716312056737588, 'f1': 0.9664902998236332, 'number': 282}","{'precision': 0.9763313609467456, 'recall': 1.0, 'f1': 0.9880239520958084, 'number': 165}","{'precision': 1.0, 'recall': 0.5, 'f1': 0.6666666666666666, 'number': 2}",0.967033,0.979955,0.973451,0.996625,0.996625,0.908406,0.996612
4,0.020900,0.013610,"{'precision': 0.9649122807017544, 'recall': 0.975177304964539, 'f1': 0.9700176366843034, 'number': 282}","{'precision': 0.9819277108433735, 'recall': 0.9878787878787879, 'f1': 0.9848942598187311, 'number': 165}","{'precision': 1.0, 'recall': 0.5, 'f1': 0.6666666666666666, 'number': 2}",0.971239,0.977728,0.974473,0.996850,0.996850,0.908227,0.996836
5,0.020900,0.013521,"{'precision': 0.9616724738675958, 'recall': 0.9787234042553191, 'f1': 0.9701230228471002, 'number': 282}","{'precision': 0.9820359281437125, 'recall': 0.9939393939393939, 'f1': 0.9879518072289156, 'number': 165}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 2}",0.969298,0.984410,0.976796,0.996963,0.996963,0.992800,0.996969


/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LEXICON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: CHRONOSTRAT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: BIOZONE seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/me

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: CHRONOSTRAT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LEXICON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: BIOZONE seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


F1 Macro: 0.8077813081265964
F1 Micro: 0.995492594977463
F1 Weighted: 0.9954055972895333
{'eval_loss': 0.02063911408185959, 'eval_EXICON': {'precision': 0.9549718574108818, 'recall': 0.9695238095238096, 'f1': 0.9621928166351607, 'number': 525}, 'eval_HRONOSTRAT': {'precision': 0.9461077844311377, 'recall': 0.9844236760124611, 'f1': 0.9648854961832062, 'number': 321}, 'eval_IOZONE': {'precision': 1.0, 'recall': 0.16666666666666666, 'f1': 0.2857142857142857, 'number': 6}, 'eval_overall_precision': 0.9516129032258065, 'eval_overall_recall': 0.9694835680751174, 'eval_overall_f1': 0.9604651162790697, 'eval_overall_accuracy': 0.995492594977463, 'eval_f1_micro': 0.995492594977463, 'eval_f1_macro': 0.8077813081265964, 'eval_f1_weighted': 0.9954055972895333, 'eval_runtime': 11.6783, 'eval_samples_per_second': 53.518, 'eval_steps_per_second': 3.425, 'epoch': 5.0}




0

In [39]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu": "0"})

In [40]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch



time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [41]:
results = {}
trainer_all = {}
s = splits[4]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(geo_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: heur_rare


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([4]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([4, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 625/625 [00:00<00:00, 10531.42 examples/s]
/tmp/ipykernel_382112/1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Exicon,Hronostrat,Iozone,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,No log,0.024384,"{'precision': 0.9444444444444444, 'recall': 0.9834710743801653, 'f1': 0.9635627530364373, 'number': 242}","{'precision': 0.8545454545454545, 'recall': 0.9463087248322147, 'f1': 0.8980891719745222, 'number': 149}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 3}",0.908873,0.961929,0.934649,0.992572,0.992572,0.722332,0.992521
2,No log,0.015485,"{'precision': 0.970954356846473, 'recall': 0.9669421487603306, 'f1': 0.9689440993788819, 'number': 242}","{'precision': 0.935064935064935, 'recall': 0.9664429530201343, 'f1': 0.9504950495049505, 'number': 149}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 3}",0.956962,0.959391,0.958175,0.995401,0.995401,0.733819,0.995237
3,No log,0.020030,"{'precision': 0.9551020408163265, 'recall': 0.9669421487603306, 'f1': 0.9609856262833676, 'number': 242}","{'precision': 0.9473684210526315, 'recall': 0.9664429530201343, 'f1': 0.9568106312292358, 'number': 149}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 3}",0.952141,0.959391,0.955752,0.995401,0.995401,0.733334,0.995240
4,0.023500,0.017040,"{'precision': 0.9713114754098361, 'recall': 0.9793388429752066, 'f1': 0.9753086419753086, 'number': 242}","{'precision': 0.9473684210526315, 'recall': 0.9664429530201343, 'f1': 0.9568106312292358, 'number': 149}","{'precision': 1.0, 'recall': 0.3333333333333333, 'f1': 0.5, 'number': 3}",0.962217,0.969543,0.965866,0.996227,0.996227,0.860239,0.996175
5,0.023500,0.017831,"{'precision': 0.9832635983263598, 'recall': 0.9710743801652892, 'f1': 0.9771309771309772, 'number': 242}","{'precision': 0.9477124183006536, 'recall': 0.9731543624161074, 'f1': 0.9602649006622517, 'number': 149}","{'precision': 1.0, 'recall': 0.6666666666666666, 'f1': 0.8, 'number': 3}",0.969543,0.969543,0.969543,0.996109,0.996109,0.935923,0.996101


/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: CHRONOSTRAT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LEXICON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: BIOZONE seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/me

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LEXICON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: CHRONOSTRAT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: BIOZONE seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


F1 Macro: 0.9024771512840177
F1 Micro: 0.9953665139525197
F1 Weighted: 0.9953883105481522
{'eval_loss': 0.017256462946534157, 'eval_EXICON': {'precision': 0.9345794392523364, 'recall': 0.9652509652509652, 'f1': 0.949667616334283, 'number': 518}, 'eval_HRONOSTRAT': {'precision': 0.9646017699115044, 'recall': 0.9849397590361446, 'f1': 0.9746646795827123, 'number': 332}, 'eval_IOZONE': {'precision': 1.0, 'recall': 0.5, 'f1': 0.6666666666666666, 'number': 2}, 'eval_overall_precision': 0.9462857142857143, 'eval_overall_recall': 0.971830985915493, 'eval_overall_f1': 0.9588882455124494, 'eval_overall_accuracy': 0.9953665139525197, 'eval_f1_micro': 0.9953665139525197, 'eval_f1_macro': 0.9024771512840177, 'eval_f1_weighted': 0.9953883105481522, 'eval_runtime': 13.8984, 'eval_samples_per_second': 44.969, 'eval_steps_per_second': 2.878, 'epoch': 5.0}




0

In [42]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu": "0"})

In [43]:
#del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch


#time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
results = {}
trainer_all = {}
s = splits[5]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(geo_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------

torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: std


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([4]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([4, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 594/594 [00:00<00:00, 12923.02 examples/s]
/tmp/ipykernel_382112/1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Exicon,Hronostrat,Iozone,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,No log,0.018151,"{'precision': 0.9598853868194842, 'recall': 0.9738372093023255, 'f1': 0.9668109668109669, 'number': 344}","{'precision': 0.975, 'recall': 0.9558823529411765, 'f1': 0.9653465346534653, 'number': 204}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 4}",0.965392,0.960145,0.962761,0.995376,0.995376,0.736888,0.995185
2,No log,0.017435,"{'precision': 0.9578651685393258, 'recall': 0.9912790697674418, 'f1': 0.9742857142857143, 'number': 344}","{'precision': 0.9753694581280788, 'recall': 0.9705882352941176, 'f1': 0.972972972972973, 'number': 204}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 4}",0.964222,0.976449,0.970297,0.995931,0.995931,0.738209,0.995758
3,No log,0.016941,"{'precision': 0.9631728045325779, 'recall': 0.9883720930232558, 'f1': 0.9756097560975608, 'number': 344}","{'precision': 0.9705882352941176, 'recall': 0.9705882352941176, 'f1': 0.9705882352941176, 'number': 204}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 4}",0.965889,0.974638,0.970243,0.996486,0.996486,0.739691,0.996306
4,0.023000,0.018253,"{'precision': 0.9550561797752809, 'recall': 0.9883720930232558, 'f1': 0.9714285714285714, 'number': 344}","{'precision': 0.9753694581280788, 'recall': 0.9705882352941176, 'f1': 0.972972972972973, 'number': 204}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 4}",0.962433,0.974638,0.968497,0.996301,0.996301,0.739752,0.996123
5,0.023000,0.018154,"{'precision': 0.9605633802816902, 'recall': 0.9912790697674418, 'f1': 0.9756795422031473, 'number': 344}","{'precision': 0.9707317073170731, 'recall': 0.9754901960784313, 'f1': 0.9731051344743276, 'number': 204}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 4}",0.962567,0.978261,0.970350,0.996578,0.996578,0.740269,0.996447


/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LEXICON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: CHRONOSTRAT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: BIOZONE seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/me

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: CHRONOSTRAT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LEXICON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


F1 Macro: 0.9741013480903137
F1 Micro: 0.9934453092994674
F1 Weighted: 0.9934881541623304
{'eval_loss': 0.02197330817580223, 'eval_EXICON': {'precision': 0.9080717488789237, 'recall': 0.9462616822429907, 'f1': 0.9267734553775744, 'number': 428}, 'eval_HRONOSTRAT': {'precision': 0.9857142857142858, 'recall': 0.9324324324324325, 'f1': 0.9583333333333333, 'number': 74}, 'eval_overall_precision': 0.9186046511627907, 'eval_overall_recall': 0.9442231075697212, 'eval_overall_f1': 0.9312377210216111, 'eval_overall_accuracy': 0.9934453092994674, 'eval_f1_micro': 0.9934453092994674, 'eval_f1_macro': 0.9741013480903137, 'eval_f1_weighted': 0.9934881541623304, 'eval_runtime': 12.9245, 'eval_samples_per_second': 45.959, 'eval_steps_per_second': 2.94, 'epoch': 5.0}




24

In [45]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu": "0"})

In [46]:
del results, trainer_all, trainer, metrics

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch

time.sleep(30)


!find . -name "*.ipynb_checkpoints" -exec rm -rf {} +

gc.collect()
torch.cuda.empty_cache()  # solta o cache ocioso do PyTorch

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [ ]:
results = {}
trainer_all = {}
s = splits[6
           ]
print(f"Treinando com split: {s}")
trainer, metrics = train_ner_with_split(geo_full, split=s)
results[s] = metrics
trainer_all[s] = trainer
print("F1 Macro:", metrics["eval_f1_macro"])
print("F1 Micro:", metrics["eval_f1_micro"])
print("F1 Weighted:", metrics["eval_f1_weighted"])
print(metrics)
print("\n")
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
# --- limpa GPU e coleta lixo -------------
torch.cuda.empty_cache()  # devolve memória à GPU
gc.collect()

Treinando com split: advs
Selecionando 625 sentenças para teste…
  0 selecionadas…
  25 selecionadas…
  46 selecionadas…
  67 selecionadas…
  92 selecionadas…
  103 selecionadas…
  119 selecionadas…
  142 selecionadas…
  165 selecionadas…
  185 selecionadas…
  202 selecionadas…
  219 selecionadas…
  234 selecionadas…
  253 selecionadas…
  272 selecionadas…
  286 selecionadas…
  301 selecionadas…
  319 selecionadas…
  331 selecionadas…
  351 selecionadas…
  369 selecionadas…
  389 selecionadas…
  405 selecionadas…
  420 selecionadas…
  435 selecionadas…
  453 selecionadas…
  466 selecionadas…
  482 selecionadas…
  496 selecionadas…
  510 selecionadas…
  527 selecionadas…
  538 selecionadas…
  547 selecionadas…
  563 selecionadas…
  575 selecionadas…
  585 selecionadas…
  598 selecionadas…
  612 selecionadas…
  620 selecionadas…


Some weights of DistilBertForTokenClassification were not initialized from the model checkpoint at Davlan/distilbert-base-multilingual-cased-ner-hrl and are newly initialized because the shapes did not match:
- classifier.bias: found shape torch.Size([9]) in the checkpoint and torch.Size([4]) in the model instantiated
- classifier.weight: found shape torch.Size([9, 768]) in the checkpoint and torch.Size([4, 768]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Map: 100%|██████████| 625/625 [00:00<00:00, 10722.69 examples/s]
/tmp/ipykernel_382112/1801789067.py:170: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss,Exicon,Hronostrat,Iozone,Overall Precision,Overall Recall,Overall F1,Overall Accuracy,F1 Micro,F1 Macro,F1 Weighted
1,No log,0.023041,"{'precision': 0.9490909090909091, 'recall': 0.9595588235294118, 'f1': 0.9542961608775138, 'number': 272}","{'precision': 0.9593908629441624, 'recall': 0.9742268041237113, 'f1': 0.9667519181585678, 'number': 194}","{'precision': 0.0, 'recall': 0.0, 'f1': 0.0, 'number': 4}",0.953390,0.957447,0.955414,0.994713,0.994713,0.734852,0.994499
2,No log,0.024995,"{'precision': 0.9458483754512635, 'recall': 0.9632352941176471, 'f1': 0.9544626593806922, 'number': 272}","{'precision': 0.9791666666666666, 'recall': 0.9690721649484536, 'f1': 0.9740932642487047, 'number': 194}","{'precision': 1.0, 'recall': 0.5, 'f1': 0.6666666666666666, 'number': 4}",0.959660,0.961702,0.960680,0.995057,0.995057,0.902658,0.995034
3,No log,0.023403,"{'precision': 0.9492753623188406, 'recall': 0.9632352941176471, 'f1': 0.9562043795620438, 'number': 272}","{'precision': 0.9644670050761421, 'recall': 0.979381443298969, 'f1': 0.9718670076726341, 'number': 194}","{'precision': 1.0, 'recall': 1.0, 'f1': 1.0, 'number': 4}",0.955975,0.970213,0.963041,0.994943,0.994943,0.985814,0.994965
4,0.019900,0.024395,"{'precision': 0.9399293286219081, 'recall': 0.9779411764705882, 'f1': 0.9585585585585585, 'number': 272}","{'precision': 0.9893617021276596, 'recall': 0.9587628865979382, 'f1': 0.9738219895287958, 'number': 194}","{'precision': 0.8, 'recall': 1.0, 'f1': 0.888888888888889, 'number': 4}",0.957983,0.970213,0.964059,0.995057,0.995057,0.958737,0.995084
5,0.019900,0.023754,"{'precision': 0.9566787003610109, 'recall': 0.9742647058823529, 'f1': 0.9653916211293261, 'number': 272}","{'precision': 0.9894179894179894, 'recall': 0.9639175257731959, 'f1': 0.9765013054830287, 'number': 194}","{'precision': 0.8, 'recall': 1.0, 'f1': 0.888888888888889, 'number': 4}",0.968153,0.970213,0.969182,0.995747,0.995747,0.960535,0.995760


/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: CHRONOSTRAT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LEXICON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: BIOZONE seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/hartb/Estudos/me

/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: LEXICON seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))
/home/hartb/Estudos/mestrado/ner_splits/lib/python3.11/site-packages/seqeval/metrics/sequence_labeling.py:171: UserWarning: CHRONOSTRAT seems not to be NE tag.
  warnings.warn('{} seems not to be NE tag.'.format(chunk))


F1 Macro: 0.9857337102872578
F1 Micro: 0.9969512195121951
F1 Weighted: 0.9969551967573308
{'eval_loss': 0.014980402775108814, 'eval_EXICON': {'precision': 0.9558823529411765, 'recall': 0.9766277128547579, 'f1': 0.9661436829066887, 'number': 599}, 'eval_HRONOSTRAT': {'precision': 0.9489795918367347, 'recall': 0.96875, 'f1': 0.9587628865979382, 'number': 96}, 'eval_overall_precision': 0.9549295774647887, 'eval_overall_recall': 0.9755395683453237, 'eval_overall_f1': 0.9651245551601423, 'eval_overall_accuracy': 0.9969512195121951, 'eval_f1_micro': 0.9969512195121951, 'eval_f1_macro': 0.9857337102872578, 'eval_f1_weighted': 0.9969551967573308, 'eval_runtime': 13.5764, 'eval_samples_per_second': 46.036, 'eval_steps_per_second': 2.946, 'epoch': 5.0}




24

In [54]:
ner_collector.record(split_name=s, metrics=metrics, extras={"gpu": "0"})

In [55]:
df = ner_collector.to_dataframes()

In [56]:
df

(       split  eval_loss  overall_precision  overall_recall  overall_f1  \
 0        loc   0.026581           0.927512        0.933665    0.930579   
 1    reverse   0.034741           0.965000        0.941463    0.953086   
 2   semantic   0.015234           0.948148        0.974619    0.961202   
 3   heur_len   0.020639           0.951613        0.969484    0.960465   
 4  heur_rare   0.017256           0.946286        0.971831    0.958888   
 5        std   0.021973           0.918605        0.944223    0.931238   
 6        std   0.022587           0.923228        0.934263    0.928713   
 7       advs   0.014980           0.954930        0.975540    0.965125   
 
    overall_accuracy  f1_micro  f1_macro  f1_weighted  runtime_s  \
 0          0.993051  0.993051  0.891938     0.993058     9.8174   
 1          0.991870  0.991870  0.858211     0.991762    15.7337   
 2          0.996144  0.996144  0.983486     0.996161    12.8165   
 3          0.995493  0.995493  0.807781     0.9954

In [57]:
nome_modelo = MODEL_NAME.split("/")[1]

In [58]:
ner_collector.to_csv(base_name=nome_modelo)

('albert-base-v2_overall_metrics.csv', 'albert-base-v2_label_metrics.csv')